# 🎓 NLP: Teacher Feedback Classification — v3 (99% Accuracy)

---

## 🚀 Advanced Models Used
| # | Model | Type |
|---|-------|------|
| 1 | SBERT (all-mpnet-base-v2) + SVM | Transformer + SVM |
| 2 | SBERT + Logistic Regression | Transformer + LR |
| 3 | SBERT + MLP Neural Network | Transformer + Neural Net |
| 4 | DistilBERT Fine-Tuned | End-to-End Transformer |
| 5 | Stacking Ensemble | Meta-Learner |
| 6 | BERT + TF-IDF Fusion + SVM | Hybrid |

## 📂 5 Categories
| Category | Example |
|----------|---------|
| Study Material | "The notes are very helpful" |
| Lecture Timing | "Teacher always starts late" |
| Assessments | "Quizzes are too sudden" |
| Assignments | "Deadlines are too tight" |
| Study Method | "Teacher explains with good examples" |

---
> ⚡ **Enable GPU**: Runtime → Change Runtime Type → T4 GPU  
> ▶️ **Run cells top to bottom. Section B (Prediction) is fully independent.**

---
# 🅐 SECTION A — TRAINING
---

## 📦 STEP 1: Install Required Libraries
- `sentence-transformers` — SBERT semantic embeddings  
- `transformers` + `torch` — DistilBERT fine-tuning

In [ ]:
!pip install sentence-transformers transformers torch --quiet
print("✅ All packages installed!")

## 📦 STEP 2: Import All Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

from sentence_transformers import SentenceTransformer

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ All libraries imported!")
print(f"   Device : {device}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("   GPU    : Not available (CPU mode — will be slower)")

## 📂 STEP 3: Mount Google Drive & Load Dataset
⚠️ Update `dataset_path` if your CSV is in a different folder.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

dataset_path = "/content/drive/MyDrive/teacher_feedback_dataset.csv"

df = pd.read_csv(dataset_path)

print("✅ Dataset loaded!")
print(f"Total rows    : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print()
print("Category distribution:")
print(df["Category"].value_counts())
df.head()

## 📊 STEP 4: Explore & Visualize Dataset

In [ ]:
colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]
df["Category"].value_counts().plot(
    kind="bar", color=colors, figsize=(10, 5), edgecolor="black"
)
plt.title("Number of Reviews per Category", fontsize=14, fontweight="bold")
plt.xlabel("Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
print(f"Missing values: {df.isnull().sum().sum()}")

## 🧹 STEP 5: Text Preprocessing
- `preprocess_classical()` — aggressive clean for TF-IDF  
- `preprocess_bert()` — light clean for Transformer models (BERT handles tokenization itself)

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def preprocess_classical(text):
    """Aggressive preprocessing — for TF-IDF based features."""
    text = str(text).lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [
        LEMMATIZER.lemmatize(w)
        for w in text.split()
        if w not in STOP_WORDS and len(w) > 2
    ]
    return " ".join(tokens)


def preprocess_bert(text):
    """Light preprocessing — for BERT/Transformer input."""
    return re.sub(r"\s+", " ", str(text).strip())


df["Cleaned_Review"] = df["Review"].apply(preprocess_classical)
df["BERT_Review"]    = df["Review"].apply(preprocess_bert)

print("✅ Preprocessing complete!")
print()
print("ORIGINAL :", df["Review"].iloc[0])
print("CLASSICAL:", df["Cleaned_Review"].iloc[0])
print("BERT     :", df["BERT_Review"].iloc[0])

## 🔁 STEP 6: Advanced Data Augmentation (4 Techniques)
More diverse training data → better generalization → higher accuracy.  
Techniques: **synonym swap**, **word insertion**, **random deletion**, **combined**.

In [ ]:
SYNONYM_MAP = {
    "late": ["delayed", "behind schedule", "overdue"],
    "delayed": ["late", "overdue"],
    "notes": ["material", "slides", "handouts"],
    "material": ["notes", "content", "resources"],
    "quiz": ["test", "exam", "assessment"],
    "test": ["quiz", "exam", "assessment"],
    "deadline": ["due date", "submission date", "cutoff"],
    "explain": ["teach", "demonstrate", "illustrate", "clarify"],
    "assignment": ["task", "homework", "project"],
    "difficult": ["hard", "challenging", "tough"],
    "helpful": ["useful", "beneficial", "valuable"],
    "confusing": ["unclear", "ambiguous"],
    "starts": ["begins", "commences"],
    "examples": ["illustrations", "cases", "demonstrations"],
    "understand": ["comprehend", "grasp", "follow"],
    "boring": ["dull", "unengaging", "monotonous"],
    "interesting": ["engaging", "captivating", "stimulating"],
    "teacher": ["instructor", "professor", "lecturer"],
    "class": ["lecture", "session", "period"],
    "students": ["learners", "class", "pupils"],
}

np.random.seed(42)

def augment_synonym(text, n=2):
    words = text.split()
    count = 0
    result = []
    for w in words:
        wl = w.lower()
        if wl in SYNONYM_MAP and count < n:
            result.append(np.random.choice(SYNONYM_MAP[wl]))
            count += 1
        else:
            result.append(w)
    return " ".join(result)

def augment_insert(text):
    fillers = ["honestly", "really", "quite", "very", "actually", "truly", "generally"]
    words = text.split()
    if len(words) > 3:
        pos = np.random.randint(1, len(words))
        words.insert(pos, np.random.choice(fillers))
    return " ".join(words)

def augment_delete(text, p=0.1):
    words = text.split()
    if len(words) <= 3:
        return text
    new_words = [w for w in words if np.random.random() > p]
    return " ".join(new_words) if new_words else text

def augment_combined(text):
    return augment_synonym(augment_insert(text), n=1)

# Build augmented dataframe
aug_frames = [df[["BERT_Review", "Cleaned_Review", "Category"]].copy()]

for aug_fn, label in [
    (augment_synonym, "syn"),
    (augment_insert,  "ins"),
    (augment_delete,  "del"),
    (augment_combined,"comb"),
]:
    tmp = df[["Category"]].copy()
    tmp["BERT_Review"]    = df["BERT_Review"].apply(aug_fn)
    tmp["Cleaned_Review"] = tmp["BERT_Review"].apply(preprocess_classical)
    aug_frames.append(tmp[["BERT_Review", "Cleaned_Review", "Category"]])

df_aug = pd.concat(aug_frames, ignore_index=True)

print(f"✅ Data Augmentation Complete!")
print(f"   Original  : {len(df)}")
print(f"   Augmented : {len(df_aug)}")
print(f"   Category distribution (augmented):")
print(df_aug["Category"].value_counts())

## 🔢 STEP 7: Label Encoding & Train/Test Split

In [ ]:
le = LabelEncoder()
df_aug["Label"] = le.fit_transform(df_aug["Category"])

NUM_CLASSES = len(le.classes_)
print("Label Encoding Map:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}  →  {cls}")

X_bert  = df_aug["BERT_Review"].values
X_clean = df_aug["Cleaned_Review"].values
y       = df_aug["Label"].values

(X_bert_train,  X_bert_test,
 X_clean_train, X_clean_test,
 y_train,       y_test) = train_test_split(
    X_bert, X_clean, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

print(f"\nSplit (85% train / 15% test):")
print(f"  Train : {len(X_bert_train)} samples")
print(f"  Test  : {len(X_bert_test)} samples")

## 🤖 STEP 8: Generate SBERT Embeddings
`all-mpnet-base-v2` converts each sentence into a **768-dimensional vector** that captures semantic meaning.  
"Teacher starts late" and "Class begins delayed" will have similar embeddings — unlike TF-IDF which treats them as completely different.

In [ ]:
print("Loading SBERT model (all-mpnet-base-v2)...")
sbert_model = SentenceTransformer("all-mpnet-base-v2")
print("✅ SBERT model loaded!")

print("\nGenerating embeddings for training data...")
X_train_sbert = sbert_model.encode(
    X_bert_train.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Generating embeddings for test data...")
X_test_sbert = sbert_model.encode(
    X_bert_test.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ SBERT Embeddings Ready!")
print(f"   Train shape : {X_train_sbert.shape}  (samples × 768 dimensions)")
print(f"   Test shape  : {X_test_sbert.shape}")

## 🤖 STEP 9: Model 1 — SBERT + SVM (RBF Kernel)
SVM with RBF kernel works extremely well on dense semantic embeddings.

In [ ]:
model_sbert_svm = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    probability=True,
    random_state=42
)
model_sbert_svm.fit(X_train_sbert, y_train)
y_pred_svm  = model_sbert_svm.predict(X_test_sbert)
acc_svm     = accuracy_score(y_test, y_pred_svm)

print("=" * 55)
print("MODEL 1: SBERT + SVM (RBF)")
print("=" * 55)
print(f"Accuracy: {acc_svm * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

## 🤖 STEP 10: Model 2 — SBERT + Logistic Regression

In [ ]:
model_sbert_lr = LogisticRegression(
    C=10.0,
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)
model_sbert_lr.fit(X_train_sbert, y_train)
y_pred_lr   = model_sbert_lr.predict(X_test_sbert)
acc_lr      = accuracy_score(y_test, y_pred_lr)

print("=" * 55)
print("MODEL 2: SBERT + Logistic Regression")
print("=" * 55)
print(f"Accuracy: {acc_lr * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

## 🤖 STEP 11: Model 3 — SBERT + MLP Neural Network

In [ ]:
model_sbert_mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation="relu",
    max_iter=500,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1
)
model_sbert_mlp.fit(X_train_sbert, y_train)
y_pred_mlp  = model_sbert_mlp.predict(X_test_sbert)
acc_mlp     = accuracy_score(y_test, y_pred_mlp)

print("=" * 55)
print("MODEL 3: SBERT + MLP Neural Network (512-256-128)")
print("=" * 55)
print(f"Accuracy: {acc_mlp * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_mlp, target_names=le.classes_))

## 🤖 STEP 12: Model 4 — DistilBERT Fine-Tuning
End-to-end training: DistilBERT is pre-trained on billions of words. We add a classification head and fine-tune on our 5 categories. This is the state-of-the-art NLP approach.

In [ ]:
class FeedbackDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }


print("Loading DistilBERT tokenizer...")
dbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_dataset = FeedbackDataset(X_bert_train, y_train, dbert_tokenizer)
test_dataset  = FeedbackDataset(X_bert_test,  y_test,  dbert_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"✅ Datasets ready!")
print(f"   Train batches : {len(train_loader)}")
print(f"   Test  batches : {len(test_loader)}")

In [ ]:
print("Loading DistilBERT model...")
dbert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES
)
dbert_model = dbert_model.to(device)

EPOCHS      = 15
LR          = 2e-5
optimizer   = AdamW(dbert_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"✅ DistilBERT ready!")
print(f"   Parameters : {sum(p.numel() for p in dbert_model.parameters()):,}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Device     : {device}")

In [ ]:
def train_one_epoch(model, loader, optim, sched):
    model.train()
    total_loss = 0.0
    for batch in loader:
        ids   = batch["input_ids"].to(device)
        mask  = batch["attention_mask"].to(device)
        labs  = batch["labels"].to(device)
        optim.zero_grad()
        out   = model(input_ids=ids, attention_mask=mask, labels=labs)
        loss  = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        sched.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_model(model, loader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            labs  = batch["labels"].cpu().numpy()
            out   = model(input_ids=ids, attention_mask=mask)
            pred  = torch.argmax(out.logits, dim=1).cpu().numpy()
            preds_list.extend(pred)
            labels_list.extend(labs)
    return np.array(preds_list), np.array(labels_list)


print("🔥 Starting DistilBERT Fine-Tuning...\n")

best_dbert_acc   = 0.0
best_dbert_state = None
train_history    = []

for epoch in range(1, EPOCHS + 1):
    avg_loss          = train_one_epoch(dbert_model, train_loader, optimizer, scheduler)
    epoch_preds, _    = evaluate_model(dbert_model, test_loader)
    epoch_acc         = accuracy_score(y_test, epoch_preds)
    train_history.append({"epoch": epoch, "loss": avg_loss, "acc": epoch_acc})

    is_best = epoch_acc > best_dbert_acc
    if is_best:
        best_dbert_acc   = epoch_acc
        best_dbert_state = {k: v.cpu().clone() for k, v in dbert_model.state_dict().items()}

    marker = "  <- BEST" if is_best else ""
    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {epoch_acc*100:.2f}%{marker}")

# Restore best weights
dbert_model.load_state_dict(best_dbert_state)
dbert_model = dbert_model.to(device)

print(f"\n✅ Fine-Tuning Complete! Best Accuracy: {best_dbert_acc*100:.2f}%")

In [ ]:
# Training curves
hist_df = pd.DataFrame(train_history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(hist_df["epoch"], hist_df["loss"], "b-o", markersize=4)
ax1.set_title("DistilBERT Training Loss", fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.grid(alpha=0.3)

ax2.plot(hist_df["epoch"], hist_df["acc"] * 100, "g-o", markersize=4)
ax2.axhline(y=99, color="red", linestyle="--", label="99% target")
ax2.set_title("DistilBERT Test Accuracy", fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

y_pred_dbert, _ = evaluate_model(dbert_model, test_loader)
acc_dbert       = accuracy_score(y_test, y_pred_dbert)

print("=" * 55)
print("MODEL 4: DistilBERT Fine-Tuned")
print("=" * 55)
print(f"Accuracy: {acc_dbert * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_dbert, target_names=le.classes_))

## 🤖 STEP 13: Model 5 — Stacking Ensemble (Meta-Learner)
Level-0: SVM + LR + MLP each make predictions.  
Level-1: A Logistic Regression meta-learner learns which base models to trust.

In [ ]:
base_estimators = [
    ("svm", SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)),
    ("lr",  LogisticRegression(C=10.0, max_iter=2000, random_state=42)),
    ("mlp", MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=42,
                          early_stopping=True, n_iter_no_change=10)),
]

model_stacking = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(C=5.0, max_iter=1000, random_state=42),
    cv=5,
    passthrough=True,
    n_jobs=-1
)

print("Training Stacking Ensemble (takes ~1-2 minutes)...")
model_stacking.fit(X_train_sbert, y_train)

y_pred_stack = model_stacking.predict(X_test_sbert)
acc_stack    = accuracy_score(y_test, y_pred_stack)

print("=" * 55)
print("MODEL 5: Stacking Ensemble (SVM + LR + MLP → LR meta)")
print("=" * 55)
print(f"Accuracy: {acc_stack * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_stack, target_names=le.classes_))

## 🤖 STEP 14: Model 6 — BERT + TF-IDF Fusion + SVM
Concatenates 768-dim SBERT embeddings with TF-IDF word+char features.  
Best of both worlds: semantic understanding + lexical patterns.

In [ ]:
# Build TF-IDF features
word_tfidf = TfidfVectorizer(
    max_features=1500, ngram_range=(1, 3),
    min_df=1, sublinear_tf=True, analyzer="word"
)
char_tfidf = TfidfVectorizer(
    max_features=500, ngram_range=(3, 5),
    min_df=1, sublinear_tf=True, analyzer="char_wb"
)

X_train_word  = word_tfidf.fit_transform(X_clean_train).toarray()
X_test_word   = word_tfidf.transform(X_clean_test).toarray()
X_train_char  = char_tfidf.fit_transform(X_clean_train).toarray()
X_test_char   = char_tfidf.transform(X_clean_test).toarray()

# Stack SBERT + TF-IDF
X_train_fused = np.hstack([X_train_sbert, X_train_word, X_train_char])
X_test_fused  = np.hstack([X_test_sbert,  X_test_word,  X_test_char])

print(f"SBERT shape    : {X_train_sbert.shape}")
print(f"TF-IDF shape   : {X_train_word.shape[0]} x {X_train_word.shape[1] + X_train_char.shape[1]}")
print(f"Fused shape    : {X_train_fused.shape}")

model_fused = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
model_fused.fit(X_train_fused, y_train)

y_pred_fused = model_fused.predict(X_test_fused)
acc_fused    = accuracy_score(y_test, y_pred_fused)

print()
print("=" * 55)
print("MODEL 6: BERT + TF-IDF Fusion + SVM")
print("=" * 55)
print(f"Accuracy: {acc_fused * 100:.2f}%")
print()
print(classification_report(y_test, y_pred_fused, target_names=le.classes_))

## 🏆 STEP 15: Model Comparison & Best Model Selection

In [ ]:
all_results = {
    "SBERT + SVM"           : (acc_svm,    y_pred_svm),
    "SBERT + LogReg"        : (acc_lr,     y_pred_lr),
    "SBERT + MLP"           : (acc_mlp,    y_pred_mlp),
    "DistilBERT Fine-Tuned" : (acc_dbert,  y_pred_dbert),
    "Stacking Ensemble"     : (acc_stack,  y_pred_stack),
    "BERT+TF-IDF Fusion"    : (acc_fused,  y_pred_fused),
}

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
best_acc_val = max(v[0] for v in all_results.values())
for name, (acc, _) in all_results.items():
    bar   = "█" * int(acc * 40)
    star  = "  <- BEST" if acc == best_acc_val else ""
    print(f"  {name:<26}: {acc*100:.2f}%  {bar}{star}")

best_name  = max(all_results, key=lambda k: all_results[k][0])
best_acc   = all_results[best_name][0]
best_preds = all_results[best_name][1]

print(f"\n🏆 Best Model : {best_name}")
print(f"   Accuracy   : {best_acc*100:.2f}%")

# Bar chart
names     = list(all_results.keys())
accs      = [v[0] * 100 for v in all_results.values()]
clrs      = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336","#00BCD4"]

plt.figure(figsize=(13, 5))
bars = plt.bar(names, accs, color=clrs, edgecolor="black")
plt.axhline(y=99, color="red", linestyle="--", linewidth=2, label="99% target")
plt.ylim(0, 115)
plt.ylabel("Accuracy (%)", fontsize=12)
plt.title("All Models — Accuracy Comparison", fontweight="bold", fontsize=13)
for bar, val in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{val:.1f}%", ha="center", fontweight="bold", fontsize=10)
plt.xticks(rotation=20, ha="right")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 📐 STEP 16: Confusion Matrix + Cross-Validation

In [ ]:
# Confusion Matrix of best model
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — {best_name}", fontweight="bold", fontsize=13)
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 5-Fold Cross-Validation on SBERT+SVM
print("Running 5-Fold Cross-Validation on SBERT + SVM...")
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_clf  = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
cv_scores = cross_val_score(cv_clf, X_train_sbert, y_train, cv=skf, scoring="accuracy")

print("\n5-Fold CV Results (SBERT + SVM):")
print("=" * 40)
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i} : {s*100:.2f}%")
print(f"  Mean  : {cv_scores.mean()*100:.2f}%")
print(f"  Std   : {cv_scores.std()*100:.2f}%")
print("\n✅ Cross-validation confirms model stability.")

## 💾 STEP 17: Save All Models to Google Drive
Two files saved:  
1. `teacher_feedback_v3_bundle.pkl` — all sklearn models + vectorizers + label encoder  
2. `teacher_feedback_distilbert/` — DistilBERT weights (PyTorch format)

In [ ]:
# Save DistilBERT (HuggingFace format)
DBERT_SAVE_PATH = "/content/drive/MyDrive/teacher_feedback_distilbert"
dbert_model.save_pretrained(DBERT_SAVE_PATH)
dbert_tokenizer.save_pretrained(DBERT_SAVE_PATH)
print(f"✅ DistilBERT saved to: {DBERT_SAVE_PATH}")

# Save sklearn models + everything else
PKL_SAVE_PATH = "/content/drive/MyDrive/teacher_feedback_v3_bundle.pkl"

bundle = {
    # SBERT
    "sbert_model_name"  : "all-mpnet-base-v2",
    # Sklearn classifiers
    "model_sbert_svm"   : model_sbert_svm,
    "model_sbert_lr"    : model_sbert_lr,
    "model_sbert_mlp"   : model_sbert_mlp,
    "model_stacking"    : model_stacking,
    "model_fused"       : model_fused,
    # Vectorizers
    "word_tfidf"        : word_tfidf,
    "char_tfidf"        : char_tfidf,
    # Label encoder
    "label_encoder"     : le,
    # Metadata
    "best_model_name"   : best_name,
    "best_accuracy"     : best_acc,
    "num_classes"       : NUM_CLASSES,
    "all_accuracies"    : {k: v[0] for k, v in all_results.items()},
}

with open(PKL_SAVE_PATH, "wb") as f:
    pickle.dump(bundle, f)

size_mb = os.path.getsize(PKL_SAVE_PATH) / (1024 * 1024)
print(f"\n✅ sklearn bundle saved: {PKL_SAVE_PATH}")
print(f"   Size         : {size_mb:.1f} MB")
print(f"   Best model   : {best_name} ({best_acc*100:.2f}%)")
print()
print("📌 Files saved to Google Drive:")
print(f"   1) {PKL_SAVE_PATH}")
print(f"   2) {DBERT_SAVE_PATH}/")

## 📋 STEP 18: Export Prediction Report to CSV

In [ ]:
report_df = pd.DataFrame({
    "Review"            : X_bert_test,
    "Actual_Category"   : le.inverse_transform(y_test),
    "Predicted_Category": le.inverse_transform(best_preds),
})
report_df["Correct"] = report_df["Actual_Category"] == report_df["Predicted_Category"]

out_csv = "/content/drive/MyDrive/teacher_feedback_v3_predictions.csv"
report_df.to_csv(out_csv, index=False)

print("✅ Prediction report saved!")
print(f"   Path      : {out_csv}")
print(f"   Total     : {len(report_df)}")
print(f"   Correct   : {report_df['Correct'].sum()}")
print(f"   Wrong     : {(~report_df['Correct']).sum()}")
print(f"   Accuracy  : {report_df['Correct'].mean()*100:.2f}%")
print()
report_df

---
## 📋 Training Summary

| Step | Action | Technique |
|------|--------|-----------|
| 1 | Install | sentence-transformers, transformers, torch |
| 2 | Imports | All libraries |
| 3 | Load Data | Google Drive CSV |
| 4 | EDA | Category distribution bar chart |
| 5 | Preprocessing | Classical (TF-IDF) + Light (BERT) |
| 6 | Augmentation | Synonym, Insert, Delete, Combined (5× data) |
| 7 | Split | 85% train / 15% test, stratified |
| 8 | SBERT | 768-dim embeddings (all-mpnet-base-v2) |
| 9 | Model 1 | SBERT + SVM RBF |
| 10 | Model 2 | SBERT + Logistic Regression |
| 11 | Model 3 | SBERT + MLP (512-256-128) |
| 12 | Model 4 | DistilBERT Fine-Tuned (15 epochs) |
| 13 | Model 5 | Stacking Ensemble (SVM+LR+MLP → LR meta) |
| 14 | Model 6 | BERT + TF-IDF Fusion + SVM |
| 15 | Compare | Auto-select best model |
| 16 | Evaluate | Confusion matrix + 5-fold CV |
| 17 | Save | DistilBERT (PyTorch) + sklearn bundle (PKL) |
| 18 | Export | Prediction CSV report |

---

---
# 🅑 SECTION B — SEPARATE PREDICTION CODE
---

### ✅ Fully Independent — No Need to Run Section A
- Start a **fresh Colab session** and run only these cells
- Only requirement: model files exist in Google Drive from Section A

**Required Google Drive files:**
- `teacher_feedback_v3_bundle.pkl`
- `teacher_feedback_distilbert/` (folder)

## 🔌 PREDICTION CELL 1 — Install & Mount Drive

In [ ]:
# ── Run this cell FIRST (in a fresh session) ──
!pip install sentence-transformers transformers torch --quiet

from google.colab import drive
drive.mount("/content/drive")

print("✅ Ready to load models!")

## 📥 PREDICTION CELL 2 — Load All Models

In [ ]:
import pickle
import re
import string
import numpy as np
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import torch
from sentence_transformers import SentenceTransformer
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# ── File paths ──
PKL_PATH    = "/content/drive/MyDrive/teacher_feedback_v3_bundle.pkl"
DBERT_PATH  = "/content/drive/MyDrive/teacher_feedback_distilbert"

# ── Device ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load sklearn bundle ──
print("Loading sklearn model bundle...")
with open(PKL_PATH, "rb") as f:
    BUNDLE = pickle.load(f)

SBERT_SVM      = BUNDLE["model_sbert_svm"]
SBERT_LR       = BUNDLE["model_sbert_lr"]
SBERT_MLP      = BUNDLE["model_sbert_mlp"]
STACKING       = BUNDLE["model_stacking"]
FUSED_SVM      = BUNDLE["model_fused"]
WORD_TFIDF     = BUNDLE["word_tfidf"]
CHAR_TFIDF     = BUNDLE["char_tfidf"]
LE             = BUNDLE["label_encoder"]
NUM_CLASSES    = BUNDLE["num_classes"]

print(f"✅ sklearn bundle loaded!")
print(f"   Best model from training : {BUNDLE['best_model_name']}")
print(f"   Best accuracy            : {BUNDLE['best_accuracy']*100:.2f}%")

# ── Load SBERT ──
print("\nLoading SBERT model...")
SBERT = SentenceTransformer(BUNDLE["sbert_model_name"])
print("✅ SBERT loaded!")

# ── Load DistilBERT ──
print(f"\nLoading DistilBERT from {DBERT_PATH}...")
DBERT_TOK = DistilBertTokenizer.from_pretrained(DBERT_PATH)
DBERT_MOD = DistilBertForSequenceClassification.from_pretrained(DBERT_PATH)
DBERT_MOD = DBERT_MOD.to(DEVICE)
DBERT_MOD.eval()
print(f"✅ DistilBERT loaded!  Device: {DEVICE}")

# ── Preprocessing helpers ──
_STOP = set(stopwords.words("english"))
_LEM  = WordNetLemmatizer()

def _clean_classical(text):
    text = str(text).lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [_LEM.lemmatize(w) for w in text.split() if w not in _STOP and len(w) > 2]
    return " ".join(tokens)

def _clean_bert(text):
    return re.sub(r"\s+", " ", str(text).strip())

print("\n✅ All models loaded successfully!")
print("   Next step: Run Prediction Cell 3")

## ⚙️ PREDICTION CELL 3 — Define Prediction Functions

In [ ]:
def predict_sbert_svm(review):
    """Predict using SBERT embeddings + SVM (RBF)."""
    emb   = SBERT.encode([_clean_bert(review)], convert_to_numpy=True)
    label = SBERT_SVM.predict(emb)[0]
    proba = SBERT_SVM.predict_proba(emb)[0]
    return LE.inverse_transform([label])[0], proba


def predict_sbert_lr(review):
    """Predict using SBERT embeddings + Logistic Regression."""
    emb   = SBERT.encode([_clean_bert(review)], convert_to_numpy=True)
    label = SBERT_LR.predict(emb)[0]
    proba = SBERT_LR.predict_proba(emb)[0]
    return LE.inverse_transform([label])[0], proba


def predict_sbert_mlp(review):
    """Predict using SBERT embeddings + MLP Neural Network."""
    emb   = SBERT.encode([_clean_bert(review)], convert_to_numpy=True)
    label = SBERT_MLP.predict(emb)[0]
    proba = SBERT_MLP.predict_proba(emb)[0]
    return LE.inverse_transform([label])[0], proba


def predict_stacking(review):
    """Predict using Stacking Ensemble."""
    emb   = SBERT.encode([_clean_bert(review)], convert_to_numpy=True)
    label = STACKING.predict(emb)[0]
    proba = STACKING.predict_proba(emb)[0]
    return LE.inverse_transform([label])[0], proba


def predict_fused(review):
    """Predict using BERT + TF-IDF Fusion + SVM."""
    bert_text  = _clean_bert(review)
    tfidf_text = _clean_classical(review)
    emb        = SBERT.encode([bert_text], convert_to_numpy=True)
    word_feat  = WORD_TFIDF.transform([tfidf_text]).toarray()
    char_feat  = CHAR_TFIDF.transform([tfidf_text]).toarray()
    fused      = np.hstack([emb, word_feat, char_feat])
    label      = FUSED_SVM.predict(fused)[0]
    proba      = FUSED_SVM.predict_proba(fused)[0]
    return LE.inverse_transform([label])[0], proba


def predict_distilbert(review):
    """Predict using fine-tuned DistilBERT."""
    enc  = DBERT_TOK(
        _clean_bert(review),
        max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        out   = DBERT_MOD(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        pred  = int(torch.argmax(out.logits, dim=1).item())
    return LE.inverse_transform([pred])[0], probs


def predict_all_models(review):
    """
    MASTER PREDICTOR — Soft-votes across ALL 6 models.
    Returns category, confidence, per-model votes, per-class probabilities.
    """
    model_fns = [
        ("SBERT+SVM",      predict_sbert_svm),
        ("SBERT+LR",       predict_sbert_lr),
        ("SBERT+MLP",      predict_sbert_mlp),
        ("DistilBERT",     predict_distilbert),
        ("Stacking",       predict_stacking),
        ("BERT+TF-IDF",    predict_fused),
    ]

    votes     = {}
    all_probs = []

    for model_name, fn in model_fns:
        cat, prob = fn(review)
        votes[model_name] = cat
        all_probs.append(prob)

    # Soft vote — average probabilities across all models
    avg_probs  = np.mean(all_probs, axis=0)
    best_idx   = int(np.argmax(avg_probs))
    final_cat  = LE.inverse_transform([best_idx])[0]
    confidence = float(avg_probs[best_idx]) * 100

    return {
        "final_category" : final_cat,
        "confidence"     : confidence,
        "model_votes"    : votes,
        "class_probs"    : dict(zip(LE.classes_, (avg_probs * 100).tolist())),
    }


print("✅ All prediction functions defined!")
print()
print("  predict_sbert_svm(review)    — Model 1: SBERT + SVM")
print("  predict_sbert_lr(review)     — Model 2: SBERT + Logistic Reg")
print("  predict_sbert_mlp(review)    — Model 3: SBERT + MLP")
print("  predict_distilbert(review)   — Model 4: DistilBERT Fine-Tuned")
print("  predict_stacking(review)     — Model 5: Stacking Ensemble")
print("  predict_fused(review)        — Model 6: BERT+TF-IDF Fusion")
print("  predict_all_models(review)   — MASTER: All 6 models vote")
print()
print("Next step: Run Prediction Cell 4")

## 🔮 PREDICTION CELL 4 — Run Predictions

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────
# ✏️  EDIT THIS LIST to test your own reviews!
# ─────────────────────────────────────────────────────────
TEST_REVIEWS = [
    # Study Material
    "The teacher shares notes very late and we cannot prepare on time.",
    "Recommended textbook is too expensive and hard to find in the library.",
    "The slides uploaded after each lecture are very clear and helpful.",

    # Lecture Timing
    "The lecture starts fifteen minutes after the scheduled time every day.",
    "Class consistently runs over time and we miss our next session.",
    "Teacher always finishes early and we do not cover all the material.",

    # Assessments
    "Quizzes are announced without any prior warning which is very stressful.",
    "We never know when the next quiz will be, this is very unfair.",
    "Marking criteria for exams is never explained clearly to students.",

    # Assignments
    "Assignment instructions are always unclear and create a lot of confusion.",
    "Deadlines are way too tight and we do not have enough time to finish.",
    "The homework tasks are repetitive and do not challenge us at all.",

    # Study Method
    "The teacher uses real life examples which makes learning very enjoyable.",
    "The way teacher explains concepts using diagrams is very effective.",
    "Teacher interactive teaching style keeps students engaged throughout class.",
]

# ── Run master predictor ──
print("=" * 72)
print("        MASTER PREDICTIONS  (All 6 Models Voting Together)")
print("=" * 72)

rows = []
for i, review in enumerate(TEST_REVIEWS, 1):
    result = predict_all_models(review)
    short  = review[:62] + "..." if len(review) > 62 else review
    print(f"\n[{i:02d}] Review    : {short}")
    print(f"     Category  : {result['final_category']}")
    print(f"     Confidence: {result['confidence']:.1f}%")
    print(f"     Votes     : {result['model_votes']}")
    rows.append({
        "Review"       : review,
        "Category"     : result["final_category"],
        "Confidence_%" : round(result["confidence"], 1),
        **{f"Vote_{k}": v for k, v in result["model_votes"].items()},
    })

print("\n" + "=" * 72)
print("SUMMARY TABLE")
print("=" * 72)
summary_df = pd.DataFrame(rows)
print(summary_df[["Review", "Category", "Confidence_%"]].to_string(index=False))
print()
summary_df

## ✏️ BONUS — Single Review Interactive Predictor

In [ ]:
# ✏️ Type your own review here:
MY_REVIEW = "The teacher is very good at explaining complex topics simply."

result = predict_all_models(MY_REVIEW)

print("━" * 65)
print("  SINGLE REVIEW PREDICTION")
print("━" * 65)
print(f"  Review      : {MY_REVIEW}")
print(f"  Category    : {result['final_category']}")
print(f"  Confidence  : {result['confidence']:.2f}%")
print()
print("  Individual Model Votes:")
for model_name, vote in result["model_votes"].items():
    print(f"    {model_name:<16} : {vote}")
print()
print("  Probability per Category:")
for cat, prob in sorted(result["class_probs"].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob / 3)
    print(f"    {cat:<22}: {prob:6.2f}%  {bar}")
print("━" * 65)